<a href="https://colab.research.google.com/github/Navya40869/edge-idps-colab/blob/main/pipeline/03_streaming_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
from collections import deque

In [2]:
packet_buffer=deque(maxlen=10)

In [3]:
blocked_ips=set()

In [4]:
CLASSES={
    0:"Normal",
    1:"DoS",
    2:"Port Scan",
    3:"Brute Force"
 }

In [5]:
def generate_live_packet(simulated_attack_type="Normal"):
    source_ip = "192.168.1.105" if simulated_attack_type != "Normal" else f"192.168.1.{random.randint(2, 50)}"

    if source_ip in blocked_ips:
        return None

    if simulated_attack_type == "Normal":
        pkt = [random.randint(100, 300), 64, 6, random.randint(800, 1200), 0, 1, 0]

    elif simulated_attack_type == "DoS":
        pkt = [random.randint(1300, 1500), 128, 17, random.randint(40000, 60000), 10, 0, 0]

    elif simulated_attack_type == "Port Scan":
        pkt = [random.randint(40, 80), 64, 6, random.randint(400, 600), 10, 0, 5]

    else:  # Brute Force
        pkt = [random.randint(400, 600), 64, 6, random.randint(7000, 9000), 5, 5, 0]

    return source_ip, pkt

In [6]:
for i in range(12):
    packet = generate_live_packet("Normal")

    if packet:
        src_ip, pkt = packet
        packet_buffer.append(pkt)

print("Buffer Size:", len(packet_buffer))
print(packet_buffer)

Buffer Size: 10
deque([[243, 64, 6, 819, 0, 1, 0], [102, 64, 6, 853, 0, 1, 0], [168, 64, 6, 1180, 0, 1, 0], [113, 64, 6, 904, 0, 1, 0], [128, 64, 6, 993, 0, 1, 0], [243, 64, 6, 807, 0, 1, 0], [206, 64, 6, 1059, 0, 1, 0], [245, 64, 6, 963, 0, 1, 0], [145, 64, 6, 1138, 0, 1, 0], [268, 64, 6, 954, 0, 1, 0]], maxlen=10)


In [7]:
from google.colab import drive

drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
df = pd.read_csv(save_path + "live_stream_output.csv")
print(f"Log file successfully verified! Total logged packets: {len(df)}")
df.head()

In [ ]:
import shutil

# Copy risk_engine.py to the shared Google Drive folder
source_file = "risk_engine.py"
destination_folder = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"

shutil.copy(source_file, destination_folder)
print(f" Successfully copied {source_file} to Google Drive!")

In [ ]:
import time
import os
import sys
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ==========================================
# 1. MOUNT DRIVE & LOAD ARTIFACTS
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"

# Append path so Python can import Member 5's risk_engine.py
sys.path.append(save_path)
from risk_engine import evaluate_risk

print("Loading saved pipeline artifacts...")
scaler = joblib.load(save_path + "scaler.pkl")
label_encoder = joblib.load(save_path + "label_encoder.pkl")
X_test = np.load(save_path + "X_test.npy")

num_features = X_test.shape[1]
num_classes = len(label_encoder.classes_)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DEFINE 1D-CNN MODEL ARCHITECTURE
# ==========================================
class EdgeIDPS1DCNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EdgeIDPS1DCNN, self).__init__()

        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

# Load 1D-CNN Weights
model = EdgeIDPS1DCNN(num_features, num_classes).to(device)
model.load_state_dict(torch.load(save_path + "edge_idps_model.pth", map_location=device))
model.eval()

print("✅ Model, Scaler, Encoder, and Risk Engine loaded successfully!")

# ==========================================
# 3. REAL-TIME STREAMING SIMULATION LOOP
# ==========================================
output_log_path = save_path + "live_stream_output.csv"

# Reset live log file with all pipeline columns
log_headers = [
    "timestamp", "packet_id", "predicted_class",
    "confidence", "risk_score", "risk_level", "action"
]
pd.DataFrame(columns=log_headers).to_csv(output_log_path, index=False)

print("\n🚀 Starting Real-Time Edge-IDPS Pipeline...")
print(f"Streaming packets and logging directly to: {output_log_path}\n")

num_packets_to_stream = 30

for packet_id in range(1, num_packets_to_stream + 1):
    # Sample a random raw feature row from test set
    sample_index = np.random.randint(0, len(X_test))
    raw_packet = X_test[sample_index].reshape(1, -1)

    # 1. AI Classification (1D-CNN)
    tensor_packet = torch.tensor(raw_packet, dtype=torch.float32).to(device)
    with torch.no_grad():
        outputs = model(tensor_packet)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    predicted_label = label_encoder.inverse_transform([predicted_idx.item()])[0]
    conf_score = confidence.item() * 100

    # 2. Fuzzy Risk Engine Evaluation (Member 5)
    risk_score, risk_level, action = evaluate_risk(predicted_label, conf_score)

    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

    # 3. Append row to CSV
    new_entry = pd.DataFrame([{
        "timestamp": timestamp,
        "packet_id": f"PKT_{packet_id:04d}",
        "predicted_class": predicted_label,
        "confidence": f"{conf_score:.2f}%",
        "risk_score": risk_score,
        "risk_level": risk_level,
        "action": action
    }])

    new_entry.to_csv(output_log_path, mode='a', header=False, index=False)

    # Print Live Terminal Stream Output
    print(f"[{timestamp}] {f'PKT_{packet_id:04d}':<8} | Class: {str(predicted_label):<20} | Conf: {conf_score:>5.1f}% | Risk: {risk_score:>4.1f}/10 ({risk_level:<8}) | Action: {action}")

    time.sleep(0.5)

print("\n✅ Streaming loop completed! Output CSV is ready for Member 4's Dashboard.")